# PadCrop and Norm

This file performs Pad-crop resizing followed by the normalization. \
This is applied to the $B_\phi$, $B_r$, $B_\theta$ and $B_{LOS}$ components of the vector magnetic field, vertical current ($Jz$), unsigned current helicity, ($USJH$) and the photospheric magnetic free energy density ($TOTPOT$).
\
\
$\textbf{NOTE:}$ this file requires the creation of the Bitmap_centroids.csv file using $\textbf{Get_Bitmap_Centroids.py}$.

In [2]:
import numpy as np
import os
import matplotlib.pyplot as plt
import cv2
from pandas import read_csv, concat
import copy
from tqdm import tqdm

from astropy.io import fits

#### General helper functions

In [3]:
# replacing nan with 0
def replace_nan(img):
    img_file = copy.copy(img)
    ind = np.where(np.isnan(img_file) == True)
    for i in range(len(ind[0])):
        img_file[ind[0][i], ind[1][i]] = 0
    return img_file 

def nan_check(img):
    if np.isnan(img).any():
        img_new = replace_nan(img)
        return img_new
    else:
        return img


# Reading a fits file from its path
def read_fits(fl_path):
    img = fits.getdata(fl_path)
    return img


# saving the final image, img to the dest_dir directory with filename fl_name in the fits format
def save_img_fits(img, dest_dir, fl_name):
    hdu = fits.PrimaryHDU(img)
    hdu.writeto(os.path.join(dest_dir, fl_name))

#### Helper functions for the Pad-crop

$\textbf{NOTE:}$\
axis = 0 => the change will be to the rows or relating to the y-axis \
axis = 1 => the change will be to the columns or relating to x-axis

In [4]:
def padding(img, axis, fin_dim):
    if axis == 0: #pad along y or rows
        pad_img = np.zeros((fin_dim, img.shape[1]))
    elif axis == 1: #pad along x or columns
        pad_img = np.zeros((img.shape[0], fin_dim))
    x, y = pad_img.shape    
    pad_img[(x - img.shape[0])//2: (x + img.shape[0])//2, (y - img.shape[1])//2:(y + img.shape[1])//2] = img 
    #:img.shape[0], :img.shape[1]] = img
    
    return pad_img

def cropping(img, axis, cent_cord, fin_dim = 896):
    val = int(fin_dim/2)
    if cent_cord - val < 0:
        #print('case1')
        cent_cord = val
        #print(cent_cord)
    elif axis*(cent_cord + val > img.shape[1]): #if the cropped image would exceed the x-length of the image
        #print('case2')
        cent_cord = img.shape[1] - val
        #print(cent_cord)
    elif (1-axis)*(cent_cord + val >img.shape[0]): #if the cropped image would exceed the y-length of the image
        #print('case3')
        cent_cord = img.shape[0] - val
        #print(cent_cord)
        
    if axis == 1:
        crop_img = img[:,cent_cord - val:cent_cord + val]
    elif axis == 0:
        crop_img = img[cent_cord - val:cent_cord + val,:]
    return crop_img

#### Helper function for the normalization

In [5]:
# Normalization for Bp, Br, Bt, LOS, Jz, USFLUX, USJH
def normalize(img, max_val, min_val):
    n_img = np.zeros(img.shape)
    p_img = np.zeros(img.shape)
    
    p_img[img > 0] = (img[img > 0] - min_val)/(max_val - min_val)
    p_img[p_img < 0] = 0
    p_img[p_img > 1] = 1
    
    n_img[img < 0] = (img[img < 0] + min_val)/(max_val - min_val)
    n_img[n_img < -1] = -1
    n_img[n_img > 0] = 0
    
    return p_img + n_img

# Normalization for TOTPOT
def log_normalize(img, max_val, min_val):
    n_img = np.zeros(img.shape)
    p_img = np.zeros(img.shape)
    
    p_img[img > 0] = (np.log(img[img > 0]) - np.log(min_val))/(np.log(max_val) - np.log(min_val))
    p_img[p_img < 0] = 0
    p_img[p_img > 1] = 1
    
    n_img[img < 0] = (np.log(img[img < 0]) + np.log(min_val))/(np.log(max_val) - np.log(min_val))
    n_img[n_img < -1] = -1
    n_img[n_img > 0] = 0
    
    return p_img + n_img

#### Loading the paths to the directories and the Bitmaps_centroid.csv file

In [32]:
all_exts = {'Bp': '.Bp.fits', 'Br': '.Br.fits', 'Bt': '.Bt.fits', 'Jz': '.jz.fits', 'USJH': '.USJH.fits',
            'LOS': '.magnetogram.fits', 'USFLUX': '.usflux.fits', 'TOTPOT': '.TOTPOT.fits'}

In [72]:
csv_fl = r"D:\PGStuff\SolarFlare\SolarData\TrialFilesToCleanNotebooks\Bitmap_centroids.csv"
df = read_csv(csv_fl, sep = ';')
cx = df['Cx']
cy = df['Cy']

In [103]:
fm_nm = 'TOTPOT'

dir_path = os.path.join(r"D:\PGStuff\SolarFlare\SolarData\SolarDataDiffSegs", fm_nm) #directory containing the file to crop
extension = all_exts[fm_nm] #extension of the file

dest_dir = os.path.join(r"D:\PGStuff\SolarFlare\SolarData\TrialFilesToCleanNotebooks", fm_nm + "_preprocessed") #directory where the cropped images will be saved 

#### List the $T_L$ (minvals) and $T_H$ (maxvals)

In [104]:
minvals = {'Bp': 100, 'Br': 100, 'Bt': 100, 'LOS': 100, 'Jz': 0, 'USJH': 0, 'TOTPOT': 1e17 }
maxvals = {'Bp': 2000, 'Br': 2000, 'Bt': 2000, 'LOS': 2000, 'Jz': 150, 'USJH': 0.5, 'TOTPOT': 1e20 }

In [105]:
min_val = minvals[fm_nm]
max_val = maxvals[fm_nm]

size = (224, 224)

In [106]:
print((min_val, max_val))

(1e+17, 1e+20)


In [107]:
if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)
    

for i in tqdm(range(len(df))):
    #print(df['FileName'][i])
    fl_path = os.path.join(dir_path, df['FileName'][i] + extension)
    img = fits.getdata(fl_path)
    
    img = nan_check(img)
    
    y, x = img.shape
    
    if (x <= 896) and (y <= 896):
        pad_img = padding(img, axis = 0, fin_dim = 896)
        pad_img = padding(pad_img, axis = 1, fin_dim = 896)
        
        assert pad_img.shape == (896, 896), "In case 1: Shape of final image is not (896,896)"
        fin_img = pad_img # fin_image is the final image, that is to be saved
    
    elif (x <= 896) and (y > 896): # row is greater and column is smaller that 896
        pad_img = padding(img, axis = 1, fin_dim = 896)
        crop_img = cropping(pad_img, axis = 0, cent_cord = cy[i])
        
        assert crop_img.shape == (896, 896), "In case 2: Shape of final image is not (896,896)"
        fin_img = crop_img
        
    elif (x > 896) and (y <= 896): #row is smaller and column is greater than 896
        pad_img = padding(img, axis = 0, fin_dim = 896)
        crop_img = cropping(pad_img, axis = 1, cent_cord = cx[i])
        
        assert crop_img.shape == (896, 896), f"In case 3: Shape of final image is not (896,896)"
        fin_img = crop_img
        
    elif (x > 896) and (y > 896):
        crop_img = cropping(img, axis = 1, cent_cord = cx[i])
        crop_img = cropping(crop_img, axis = 0, cent_cord = cy[i])
        
        assert crop_img.shape == (896, 896), f"In case 4: Shape of final image is not (896,896);b{fl_path}"
        fin_img = crop_img
    
    #norm_img = log_normalize(fin_img, max_val, min_val)
    if fm_nm == 'TOTPOT':
        norm_img = log_normalize(fin_img, max_val, min_val)
    else:
        norm_img = normalize(fin_img, max_val, min_val)
#     norm_img = fin_img
    
    resized_img = cv2.resize(norm_img, size, interpolation = cv2.INTER_LINEAR)
    save_img_fits(resized_img, dest_dir, df['FileName'][i]+extension)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 10.73it/s]
